## Data Cleaning

This section prepares the dataset for exploratory analysis by addressing missing values, invalid records, duplicate rows, and inconsistent column structure.

# Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/raw/vgsales.csv")

In [3]:
df.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,NA_Units,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,0.0,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,0.0,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,0.0,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,0.0,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,0.0,8.89,10.22,1.00,31.37


## Data Cleaning

In [4]:
# Check for data types, missing values, and unique values
df_clean = df.copy()

quality_summary = pd.DataFrame({
    'datatype': df_clean.dtypes,
    'missing_count': df_clean.isna().sum(),
    'missing_pct': (df_clean.isna().mean() * 100).round(2),
    'number_unique': df_clean.nunique(dropna=True)
}).sort_values(by='missing_count', ascending=False)

quality_summary

,datatype,missing_count,missing_pct,number_unique
Other_Sales,float64,577,3.48,157
Year,float64,271,1.63,40
Publisher,str,76,0.46,578
Platform,str,22,0.13,31
Name,str,17,0.10,11481
Genre,str,16,0.10,12
EU_Sales,float64,7,0.04,305
JP_Sales,float64,7,0.04,244
Global_Sales,float64,7,0.04,623
NA_Units,float64,6,0.04,1


### Drop unused columns

#### Inspect columns that might be dropped 

In [5]:
# Inspect NA_Units column
df_clean['NA_Units'].describe()

count    16594.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: NA_Units, dtype: float64

In [6]:
# Drop NA_Units column due to high missing values
df_clean = df_clean.drop(columns=['NA_Units'])

In [7]:
# Drop ranking column as it is not needed for analysis
df_clean = df_clean.drop(columns=['Rank'])

### Remove duplicates

In [8]:
# Check for duplicates before dropping them
duplicates = df_clean.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

Number of duplicate rows: 3


In [9]:
# Remove duplicates
df_clean = df_clean.drop_duplicates()

### Handle missing values in categorical fields

In [10]:
# Inspect missing categorical names
df_clean[['Name', 'Platform', 'Genre', 'Publisher']].isna().sum()

Name         17
Platform     21
Genre        16
Publisher    76
dtype: int64

Name, Platform, and Genre are central to analysis. Rows missing provide little values, so they are removed.
Empty Publisher values can be replaced with "Unknown"

In [11]:
# Drop rows missing core identifying information
df_clean = df_clean.dropna(subset=['Name', 'Platform', 'Genre'])

In [12]:
# Fill missing values in 'Publisher' with 'Unknown'
df_clean['Publisher'] = df_clean['Publisher'].fillna('Unknown')

### Resolve missing sales values

In [13]:
# Check how many rows exist at the moment
print(f"Number of rows after cleaning: {len(df_clean)}")

Number of rows after cleaning: 16559


In [14]:
# Create sales columns subset
sales_columns = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']

In [15]:
# Count missing rows to determine cleaning method
missing_sales_rows = df_clean[sales_columns].isna().any(axis=1).sum()
missing_sales_percent = (missing_sales_rows / len(df_clean)) * 100
print(f"Number of rows with missing sales data: {missing_sales_rows} ({missing_sales_percent:.2f}%)")

Number of rows with missing sales data: 568 (3.43%)


Other_sales is missing by far the most values (n=568). In total, these missing values (out of ~16,600 total) is only 3.4% of the entire dataset. The simple solution of dropping these rows is best here (instead of imputing).

In [16]:
# Drop rows with missing sales data
df_clean = df_clean.dropna(subset=sales_columns)

### Validate and clean the Year column

In [17]:
# Check datatype of Year column
df_clean['Year'].dtype

dtype('float64')

In [18]:
# Change datatype of 'Year' to integer
df_clean['Year'] = pd.to_numeric(df_clean['Year'], errors='coerce').astype('Int64')

In [19]:
# Check the range of years in the cleaned dataset
print(f"Year range: {df_clean['Year'].min()} - {df_clean['Year'].max()}")

Year range: 1980 - 2017


In [20]:
# Check for na values in Year column after conversion
year_na_count = df_clean['Year'].isna().sum()


In [21]:
# Define and count invalid years (before 1980, after 2016, or missing)
invalid_years = df_clean[(df_clean['Year'] < 1980) | (df_clean['Year'] > 2016) | (df_clean['Year'].isna())]
invalid_years_count = len(invalid_years)
print(f"Number of rows with invalid Year values: {invalid_years_count}")

Number of rows with invalid Year values: 267


In [22]:
# Remove rows with invalid Year values
df_clean = df_clean[~((df_clean['Year'] < 1980) | (df_clean['Year'] > 2016) | (df_clean['Year'].isna()))]

### Final validation checks

In [23]:
# Recheck missing values
print(df_clean.isna().sum())

Name            0
Platform        0
Year            0
Genre           0
Publisher       0
NA_Sales        0
EU_Sales        0
JP_Sales        0
Other_Sales     0
Global_Sales    0
dtype: int64


In [24]:
# Recheck duplicates
df_clean.duplicated().sum()

np.int64(0)

In [25]:
# Recheck year range
df_clean['Year'].min(), df_clean['Year'].max()

(np.int64(1980), np.int64(2016))

In [26]:
# Sales consistency check
df_clean['Total_Sales_Check'] = (df_clean['NA_Sales'] + df_clean['EU_Sales'] + df_clean['JP_Sales'] + df_clean['Other_Sales'])
df_clean['Sales_Difference'] = df_clean['Global_Sales'] - df_clean['Total_Sales_Check']
df_clean['Sales_Difference'].describe()

count    15724.000000
mean         0.000260
std          0.005276
min         -0.020000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.020000
Name: Sales_Difference, dtype: float64

Mean sales difference between individual sales columns and global total is 0.000260. Negligable and acceptable

In [27]:
# Drop consistency check columns
df_clean = df_clean.drop(columns=['Total_Sales_Check', 'Sales_Difference'])

In [28]:
# Final check of cleaned dataset
df_clean.head()

,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


### Save cleaned dataset

In [29]:
# Export cleaned dataset to new CSV file
df_clean.to_csv("../data/processed/vgsales_clean.csv", index=False)